# Nykaa Domain Support Agent — Task 4: Grounded Retrieval & MOCK_LLM

This notebook implements Task 4 using the persistent ChromaDB collections created in Task 3.

It will:
1. Load the embedding model and ChromaDB collections.
2. Measure top-1 cosine similarity for at least 5 in-scope calibration queries and 2 out-of-scope queries.
3. Derive a threshold from the measured scores (not a tutorial default).
4. Retrieve the top-k chunks and produce a deterministic extractive answer using only retrieved context.
5. Demonstrate at least 5 in-scope questions and 1 out-of-scope question.

**Run this notebook from the same project directory that contains `knowledge_base/` and `chroma_db/`.**

## 1. Imports and configuration

For this first implementation, we use the fixed-size collection. Task 5 will compare the two collections and recommend which one to deploy. You can change `COLLECTION_NAME` to the sentence-based collection if needed.

In [ ]:
from pathlib import Path
import re

import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_DIR = Path.cwd() / "chroma_db"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

COLLECTION_NAME = "nykaa_fixed_chunks"
TOP_K = 3

if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        "chroma_db folder not found. Run Task 3 first and use the same project directory."
    )

embedding_model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name=COLLECTION_NAME)

print("Loaded collection:", COLLECTION_NAME)
print("Number of indexed chunks:", collection.count())


## 2. Create a retrieval function

The embedding model converts a question into a vector. ChromaDB retrieves the closest chunks. Since the Task 3 collection uses cosine distance, cosine similarity is calculated as `1 - distance`.

In [ ]:
def retrieve_chunks(query, top_k=TOP_K):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    retrieved = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": document,
            "source": metadata["source"],
            "similarity": 1 - float(distance)
        })

    return retrieved


# Quick retrieval check
quick_test = retrieve_chunks("How long does a cash-on-delivery refund take?")

for rank, item in enumerate(quick_test, start=1):
    print(f"Rank {rank} | {item['source']} | similarity={item['similarity']:.4f}")


## 3. Prepare calibration queries

Use questions that the knowledge base can answer and questions that are deliberately unrelated to its policy topics. The out-of-scope examples below are about astronomy and cooking; they should not be answered by the Nykaa policy knowledge base.

The query labels are used only to evaluate retrieval. They are not supplied to the retrieval model.

In [ ]:
IN_SCOPE_CALIBRATION = [
    "How many business days does a COD refund take?",
    "What is the return window for footwear?",
    "How long does standard domestic delivery usually take?",
    "When can a customer request a size exchange?",
    "What should a customer do if payment was deducted but no order was created?"
]

OUT_OF_SCOPE_CALIBRATION = [
    "Explain how black holes form in distant galaxies.",
    "What ingredients are needed to bake a chocolate cake?"
]

calibration_queries = (
    [(q, "in_scope") for q in IN_SCOPE_CALIBRATION]
    + [(q, "out_of_scope") for q in OUT_OF_SCOPE_CALIBRATION]
)

print("In-scope calibration queries:", len(IN_SCOPE_CALIBRATION))
print("Out-of-scope calibration queries:", len(OUT_OF_SCOPE_CALIBRATION))


## 4. Measure top-1 cosine similarity

For every calibration query, record the similarity of the single highest-ranked retrieved chunk. These measured values are the evidence used to choose the fallback threshold.

In [ ]:
calibration_results = []

for query, label in calibration_queries:
    results = retrieve_chunks(query, top_k=TOP_K)

    top1_similarity = results[0]["similarity"] if results else float("-inf")
    top1_source = results[0]["source"] if results else "No result"

    calibration_results.append({
        "query": query,
        "label": label,
        "top1_similarity": top1_similarity,
        "top1_source": top1_source
    })

for row in calibration_results:
    print(
        f"Label: {row['label']:10s} | "
        f"Top-1 cosine similarity: {row['top1_similarity']:.4f} | "
        f"Source: {row['top1_source']}\n"
        f"Query: {row['query']}\n"
    )


## 5. Calibrate the fallback threshold from observed scores

A clean separation exists only if the lowest in-scope score is greater than the highest out-of-scope score. If the two groups overlap, a single similarity threshold cannot reliably separate them. In that case, the notebook stops rather than silently choosing an arbitrary value.

When the groups are separated, the threshold is the midpoint between the two observed boundary scores.

In [ ]:
in_scores = [
    row["top1_similarity"]
    for row in calibration_results
    if row["label"] == "in_scope"
]

out_scores = [
    row["top1_similarity"]
    for row in calibration_results
    if row["label"] == "out_of_scope"
]

if len(in_scores) < 3 or len(out_scores) < 2:
    raise ValueError("Need at least 3 in-scope and 2 out-of-scope calibration queries.")

min_in_scope = min(in_scores)
max_out_of_scope = max(out_scores)

print(f"Lowest in-scope similarity: {min_in_scope:.4f}")
print(f"Highest out-of-scope similarity: {max_out_of_scope:.4f}")

if min_in_scope <= max_out_of_scope:
    raise ValueError(
        "The measured score groups overlap. Do not choose an arbitrary threshold. "
        "Inspect the queries and retrieved chunks, improve the knowledge base or "
        "retrieval setup, then rerun calibration."
    )

SIMILARITY_THRESHOLD = (min_in_scope + max_out_of_scope) / 2

print(f"Empirically calibrated threshold: {SIMILARITY_THRESHOLD:.4f}")


## 6. Implement deterministic MOCK_LLM grounded generation

This capstone must work without an API key. Therefore, this mock generator uses an extractive strategy: it returns the retrieved policy text as evidence rather than inventing new policy details.

The answer is refused if the top-1 similarity is below the empirically calibrated threshold. The returned answer is constructed only from retrieved chunks.

In [ ]:
FALLBACK_ANSWER = (
    "I don't know based on the available policy documents. "
    "Please contact customer support for confirmation."
)

def mock_llm_generate(query, retrieved_chunks):
    # This deterministic mock uses only retrieved context.
    # It does not call an external language model.
    if not retrieved_chunks:
        return FALLBACK_ANSWER

    context_parts = []
    seen = set()

    for item in retrieved_chunks:
        text = item["text"].strip()
        if text and text not in seen:
            context_parts.append(text)
            seen.add(text)

    if not context_parts:
        return FALLBACK_ANSWER

    return "According to the available project policy: " + " ".join(context_parts)


def answer_question(query, top_k=TOP_K):
    retrieved = retrieve_chunks(query, top_k=top_k)

    if not retrieved:
        return {
            "query": query,
            "answered": False,
            "similarity": None,
            "threshold": SIMILARITY_THRESHOLD,
            "answer": FALLBACK_ANSWER,
            "sources": []
        }

    top1_similarity = retrieved[0]["similarity"]

    if top1_similarity < SIMILARITY_THRESHOLD:
        return {
            "query": query,
            "answered": False,
            "similarity": top1_similarity,
            "threshold": SIMILARITY_THRESHOLD,
            "answer": FALLBACK_ANSWER,
            "sources": [item["source"] for item in retrieved]
        }

    answer = mock_llm_generate(query, retrieved)

    return {
        "query": query,
        "answered": True,
        "similarity": top1_similarity,
        "threshold": SIMILARITY_THRESHOLD,
        "answer": answer,
        "sources": [item["source"] for item in retrieved]
    }


## 7. Demonstrate at least 5 in-scope questions

These questions cover different topics from the policy knowledge base. The result shows whether the agent answered, the top-1 similarity, the threshold, the retrieved sources, and the answer.

In [ ]:
IN_SCOPE_DEMO_QUERIES = [
    "How long does a COD refund take?",
    "Can I return a beauty product after delivery?",
    "What is the standard delivery time for domestic orders?",
    "What are the conditions for exchanging footwear size?",
    "What should I do if my payment was deducted but my order was not created?"
]

for query in IN_SCOPE_DEMO_QUERIES:
    result = answer_question(query)

    print("\n" + "=" * 75)
    print("QUESTION:", result["query"])
    print("ANSWERED:", result["answered"])
    print("TOP-1 SIMILARITY:", (
        f"{result['similarity']:.4f}"
        if result["similarity"] is not None else "No result"
    ))
    print("THRESHOLD:", f"{result['threshold']:.4f}")
    print("SOURCES:", sorted(set(result["sources"])))
    print("ANSWER:", result["answer"])


## 8. Demonstrate an out-of-scope question and fallback

This deliberately unrelated question must trigger the fallback if its top-1 similarity is below the calibrated threshold.

In [ ]:
OUT_OF_SCOPE_DEMO_QUERY = (
    "Can you explain the process of photosynthesis in green plants?"
)

out_of_scope_result = answer_question(OUT_OF_SCOPE_DEMO_QUERY)

print("QUESTION:", out_of_scope_result["query"])
print("ANSWERED:", out_of_scope_result["answered"])
print("TOP-1 SIMILARITY:", (
    f"{out_of_scope_result['similarity']:.4f}"
    if out_of_scope_result["similarity"] is not None else "No result"
))
print("THRESHOLD:", f"{out_of_scope_result['threshold']:.4f}")
print("ANSWER:", out_of_scope_result["answer"])

assert out_of_scope_result["answered"] is False, (
    "The out-of-scope demonstration did not trigger fallback. "
    "Review calibration queries and retrieval scores."
)

print("\nPASS: Out-of-scope query triggered the fallback.")


## 9. Save calibration evidence for the README

This cell saves the measured scores and chosen threshold as a JSON file. Use the actual values produced by your run in the repository README.

In [ ]:
import json

calibration_report = {
    "collection": COLLECTION_NAME,
    "embedding_model": MODEL_NAME,
    "top_k": TOP_K,
    "calibration_queries": calibration_results,
    "lowest_in_scope_similarity": min_in_scope,
    "highest_out_of_scope_similarity": max_out_of_scope,
    "chosen_threshold": SIMILARITY_THRESHOLD,
    "threshold_method": "midpoint between lowest in-scope and highest out-of-scope top-1 cosine similarity"
}

report_path = Path.cwd() / "task4_threshold_calibration.json"
report_path.write_text(
    json.dumps(calibration_report, indent=2),
    encoding="utf-8"
)

print("Calibration report saved to:", report_path.resolve())
print("\nPASS: Task 4 calibration evidence saved.")
